# §1.6 アファイン接続と共変微分 - 曲がった空間での微分

## 1. 概要

- **この節で学ぶこと**: アファイン接続、共変微分、平行移動、曲率
- **前提知識**: ベクトル場、リーマン計量、Christoffel記号
- **情報幾何との関連**: **α-接続、e-接続とm-接続の双対性**

## 2. 直感的理解

### なぜ「普通の微分」ではダメなのか

- ユークリッド空間: ベクトルを平行移動しても成分は変わらない
- 曲がった空間: 基底自体が場所によって変わる
- → ベクトルの「本当の変化」と「座標系の変化」を区別する必要

### 共変微分のイメージ

- 「座標系の変化を差し引いた、真のベクトル場の変化率」
- 平行移動: 共変微分がゼロとなる移動

### 地球表面での例え

- 北極から赤道へベクトル（東向き）を平行移動
- 経路によって最終的なベクトルが異なる！
- これが「曲率」の表れ

### 情報幾何での重要性

- **α-接続**: パラメータ α で特徴づけられる接続の族
- **e-接続** (α=1): 指数型分布族で自然な接続
- **m-接続** (α=-1): 混合族で自然な接続
- **双対性**: e-接続と m-接続はFisher計量に関して双対

## 3. 数学的定義

### 3.1 アファイン接続の定義

多様体 $M$ 上の**アファイン接続** $\nabla$ とは、ベクトル場 $X, Y$ に対して
$$\nabla: \mathfrak{X}(M) \times \mathfrak{X}(M) \to \mathfrak{X}(M)$$
$$\nabla: (X, Y) \mapsto \nabla_X Y$$
で、以下を満たすもの:

1. $\nabla_{fX+gY} Z = f\nabla_X Z + g\nabla_Y Z$
2. $\nabla_X (Y + Z) = \nabla_X Y + \nabla_X Z$
3. $\nabla_X (fY) = (Xf)Y + f\nabla_X Y$ (ライプニッツ則)

### 3.2 Christoffel記号

座標基底 $\{\partial_i\}$ に対する接続係数:
$$\nabla_{\partial_i} \partial_j = \Gamma^k_{ij} \partial_k$$

ベクトル場 $Y = Y^j \partial_j$ の共変微分:
$$\nabla_X Y = X^i \left( \frac{\partial Y^k}{\partial x^i} + \Gamma^k_{ij} Y^j \right) \partial_k$$

### 3.3 Levi-Civita接続

リーマン多様体で唯一の、以下を満たす接続:
- **計量整合性**: $Xg(Y, Z) = g(\nabla_X Y, Z) + g(Y, \nabla_X Z)$
- **捩れなし**: $\nabla_X Y - \nabla_Y X = [X, Y]$

Christoffel記号の公式:
$$\Gamma^k_{ij} = \frac{1}{2} g^{kl} \left( \frac{\partial g_{il}}{\partial x^j} + \frac{\partial g_{jl}}{\partial x^i} - \frac{\partial g_{ij}}{\partial x^l} \right)$$

### 3.4 α-接続（情報幾何）

Fisher計量 $g$ に対して、**α-接続** $\nabla^{(\alpha)}$ を定義:
$$\Gamma^{(\alpha)k}_{ij} = \Gamma^{(0)k}_{ij} - \frac{\alpha}{2} T_{ij}^k$$

ここで $\Gamma^{(0)}$ はLevi-Civita接続、$T$ は歪度テンソル。

重要な特殊ケース:
- $\alpha = 0$: Levi-Civita接続
- $\alpha = 1$: **e-接続**（指数接続）
- $\alpha = -1$: **m-接続**（混合接続）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import odeint

plt.rcParams['figure.figsize'] = (10, 8)

class AffineConnection:
    """アファイン接続のクラス"""
    def __init__(self, christoffel_func, dim=2):
        """
        christoffel_func: 点 p での Γ^k_{ij} を返す関数
                          christoffel_func(p) -> array of shape (dim, dim, dim)
        """
        self.christoffel_func = christoffel_func
        self.dim = dim
    
    def christoffel(self, p):
        """点 p での Christoffel 記号 Γ^k_{ij}"""
        return self.christoffel_func(p)
    
    def covariant_derivative(self, p, X, Y, dY):
        """
        共変微分 (∇_X Y)^k = X^i (∂Y^k/∂x^i + Γ^k_{ij} Y^j)
        
        p: 点
        X: 方向ベクトル
        Y: ベクトル場の値
        dY: ベクトル場の偏微分 (∂Y^k/∂x^i)
        """
        Gamma = self.christoffel(p)
        result = np.zeros(self.dim)
        
        for k in range(self.dim):
            for i in range(self.dim):
                result[k] += X[i] * dY[i, k]
                for j in range(self.dim):
                    result[k] += X[i] * Gamma[k, i, j] * Y[j]
        
        return result
    
    def geodesic_ode(self, state, t):
        """測地線の微分方程式"""
        p = state[:self.dim]
        v = state[self.dim:]
        
        Gamma = self.christoffel(p)
        
        dp = v
        dv = np.zeros(self.dim)
        
        for k in range(self.dim):
            for i in range(self.dim):
                for j in range(self.dim):
                    dv[k] -= Gamma[k, i, j] * v[i] * v[j]
        
        return np.concatenate([dp, dv])
    
    def geodesic(self, p0, v0, t_span, n_points=100):
        """測地線を計算"""
        t = np.linspace(t_span[0], t_span[1], n_points)
        state0 = np.concatenate([p0, v0])
        
        solution = odeint(self.geodesic_ode, state0, t)
        
        return t, solution[:, :self.dim], solution[:, self.dim:]

## 4. 可視化

### 4.1 球面上の平行移動

In [ ]:
def visualize_parallel_transport_sphere():
    """Visualize parallel transport on a sphere"""
    fig = plt.figure(figsize=(12, 5))
    
    # Draw sphere
    u = np.linspace(0, 2*np.pi, 50)
    v = np.linspace(0, np.pi, 30)
    U, V = np.meshgrid(u, v)
    X = np.sin(V) * np.cos(U)
    Y = np.sin(V) * np.sin(U)
    Z = np.cos(V)
    
    # Left: 3D sphere
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.plot_surface(X, Y, Z, alpha=0.3, color='cyan')
    
    # Path 1: North pole -> equator at lon=0 -> North pole at lon=90
    # Path 2: North pole -> equator at lon=90 -> North pole at lon=0 (reverse)
    
    # Plot path (triangle)
    # North pole
    p_north = np.array([0, 0, 1])
    # Equator (longitude 0)
    p_eq_0 = np.array([1, 0, 0])
    # Equator (longitude 90)
    p_eq_90 = np.array([0, 1, 0])
    
    # Geodesics (great circles)
    t = np.linspace(0, np.pi/2, 50)
    
    # North pole -> Equator(0)
    path1_x = np.sin(t)
    path1_y = np.zeros_like(t)
    path1_z = np.cos(t)
    ax1.plot(path1_x, path1_y, path1_z, 'r-', linewidth=3, label='Path 1')
    
    # Equator(0) -> Equator(90)
    path2_x = np.cos(t)
    path2_y = np.sin(t)
    path2_z = np.zeros_like(t)
    ax1.plot(path2_x, path2_y, path2_z, 'g-', linewidth=3, label='Path 2')
    
    # Equator(90) -> North pole
    path3_x = np.zeros_like(t)
    path3_y = np.sin(np.pi/2 - t)
    path3_z = np.cos(np.pi/2 - t)
    ax1.plot(path3_x, path3_y, path3_z, 'b-', linewidth=3, label='Path 3')
    
    # Mark vertices
    ax1.scatter([0, 1, 0], [0, 0, 1], [1, 0, 0], s=100, c='black')
    
    # Arrow showing parallel transported vector
    # Initial vector at north pole (longitude 0 direction = +x direction)
    ax1.quiver(0, 0, 1, 0.3, 0, 0, color='red', arrow_length_ratio=0.3, linewidth=2)
    
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Triangular Path on Sphere\n(Geodesic Triangle)')
    ax1.legend()
    
    # Right: Result of parallel transport
    ax2 = fig.add_subplot(122)
    
    ax2.text(0.1, 0.9, '[Parallel Transport Result]', fontsize=14, transform=ax2.transAxes)
    ax2.text(0.1, 0.8, 'Initial vector: Eastward (+x) at north pole', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.1, 0.7, '', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.1, 0.6, 'Path 1 (North pole -> Equator(0)):', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.15, 0.55, '-> Southward at equator', fontsize=10, transform=ax2.transAxes)
    ax2.text(0.1, 0.45, 'Path 2 (Along equator to east):', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.15, 0.4, '-> Still southward at Equator(90)', fontsize=10, transform=ax2.transAxes)
    ax2.text(0.1, 0.3, 'Path 3 (Equator(90) -> North pole):', fontsize=11, transform=ax2.transAxes)
    ax2.text(0.15, 0.25, '-> +y direction at north pole', fontsize=10, transform=ax2.transAxes)
    ax2.text(0.1, 0.1, 'Result: Vector rotated 90 degrees!', fontsize=12, color='red', 
             fontweight='bold', transform=ax2.transAxes)
    ax2.text(0.1, 0.02, '(This manifests the curvature of the sphere)', fontsize=11, transform=ax2.transAxes)
    
    ax2.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("[Holonomy]")
    print("Parallel transport along a closed loop may not return the vector to its original direction.")
    print("Rotation angle = Area enclosed by path x Gaussian curvature")
    print(f"This example: Rotation angle = (1/8 x 4*pi) x 1 = pi/2")

visualize_parallel_transport_sphere()

### 4.2 正規分布多様体での測地線

In [ ]:
def visualize_gaussian_geodesics():
    """Geodesics on the Gaussian distribution manifold (Levi-Civita connection)"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Fisher metric and Christoffel symbols for Gaussian
    # g = diag(1/sigma^2, 2/sigma^2)
    
    def christoffel_gaussian(p):
        """Christoffel symbols for Gaussian (Levi-Civita connection)"""
        mu, sigma = p
        sigma = max(sigma, 0.1)  # For stability
        
        Gamma = np.zeros((2, 2, 2))  # Gamma^k_{ij}
        
        # Non-zero components
        Gamma[0, 0, 1] = -1/sigma  # Gamma^mu_{mu,sigma}
        Gamma[0, 1, 0] = -1/sigma  # Gamma^mu_{sigma,mu}
        Gamma[1, 0, 0] = 1/sigma   # Gamma^sigma_{mu,mu}
        Gamma[1, 1, 1] = -1/sigma  # Gamma^sigma_{sigma,sigma}
        
        return Gamma
    
    conn = AffineConnection(christoffel_gaussian)
    
    # Left: Geodesics from various initial conditions
    ax1 = axes[0]
    
    # Fisher metric ellipses as background
    theta = np.linspace(0, 2*np.pi, 100)
    for mu_c in np.linspace(-1, 2, 4):
        for sigma_c in [0.5, 1.0, 1.5]:
            scale = 0.1
            ellipse_x = sigma_c * np.cos(theta) * scale + mu_c
            ellipse_y = sigma_c / np.sqrt(2) * np.sin(theta) * scale + sigma_c
            ax1.plot(ellipse_x, ellipse_y, color='gray', linewidth=0.5, alpha=0.3)
    
    # Compute geodesics
    initial_conditions = [
        ([0, 1], [1, 0]),      # mu direction
        ([0, 1], [0, 0.5]),    # sigma direction
        ([0, 1], [1, 0.3]),    # diagonal
        ([0, 1], [1, -0.3]),   # diagonal (downward)
        ([-1, 0.8], [1, 0.2]),
    ]
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(initial_conditions)))
    
    for (p0, v0), color in zip(initial_conditions, colors):
        try:
            t, path, vel = conn.geodesic(np.array(p0), np.array(v0), [0, 2], n_points=100)
            # Plot only where sigma > 0
            valid = path[:, 1] > 0.1
            ax1.plot(path[valid, 0], path[valid, 1], color=color, linewidth=2)
            ax1.plot(p0[0], p0[1], 'o', color=color, markersize=8)
            ax1.arrow(p0[0], p0[1], v0[0]*0.2, v0[1]*0.2, head_width=0.05, 
                      head_length=0.02, fc=color, ec=color)
        except:
            pass
    
    ax1.set_xlabel('mu')
    ax1.set_ylabel('sigma')
    ax1.set_title('Geodesics on Gaussian Manifold\n(Levi-Civita Connection)')
    ax1.set_xlim(-2, 3)
    ax1.set_ylim(0.1, 2)
    ax1.grid(True, alpha=0.3)
    
    # Right: Geodesic between two points
    ax2 = axes[1]
    
    # Specify two points
    p1 = np.array([0, 1])
    p2 = np.array([2, 0.5])
    
    ax2.plot(p1[0], p1[1], 'go', markersize=15, label='Start: N(0, 1^2)')
    ax2.plot(p2[0], p2[1], 'ro', markersize=15, label='End: N(2, 0.5^2)')
    
    # Straight line (Euclidean)
    t_line = np.linspace(0, 1, 50)
    line = p1[:, None] + t_line * (p2 - p1)[:, None]
    ax2.plot(line[0], line[1], 'b--', linewidth=2, label='Straight line (Euclidean)')
    
    # Geodesic (adjust initial velocity to reach endpoint)
    v0_approx = np.array([1.5, -0.3])  # Approximate initial velocity
    t, path, vel = conn.geodesic(p1, v0_approx, [0, 1.5], n_points=100)
    valid = path[:, 1] > 0.1
    ax2.plot(path[valid, 0], path[valid, 1], 'r-', linewidth=2, label='Geodesic (Fisher metric)')
    
    ax2.set_xlabel('mu')
    ax2.set_ylabel('sigma')
    ax2.set_title('Straight Line vs Geodesic\n(Geodesic avoids small sigma region)')
    ax2.set_xlim(-0.5, 3)
    ax2.set_ylim(0.1, 1.5)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("[Observation]")
    print("- Geodesics minimize Fisher distance")
    print("- Small sigma region has large Fisher information -> same change feels 'far'")
    print("- Therefore geodesics detour around small sigma regions")

visualize_gaussian_geodesics()

### 4.3 α-接続の違い（e-接続 vs m-接続）

In [ ]:
def visualize_alpha_connections():
    """Differences between geodesics for various alpha-connections"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Alpha-connections for exponential family (Gaussian)
    # alpha = 1: e-geodesic (straight in natural parameters)
    # alpha = 0: Levi-Civita geodesic
    # alpha = -1: m-geodesic (straight in expectation parameters)
    
    # Consider Bernoulli distribution (1D)
    # p in (0, 1)
    # Natural parameter: theta = log(p/(1-p))
    # Expectation parameter: eta = p
    
    def p_to_theta(p):
        """Expectation -> natural parameter"""
        p = np.clip(p, 0.01, 0.99)
        return np.log(p / (1 - p))
    
    def theta_to_p(theta):
        """Natural parameter -> expectation"""
        return 1 / (1 + np.exp(-theta))
    
    # Two points: p1 = 0.2, p2 = 0.8
    p1, p2 = 0.2, 0.8
    theta1, theta2 = p_to_theta(p1), p_to_theta(p2)
    
    t = np.linspace(0, 1, 100)
    
    # Left: Expectation parameter space
    ax1 = axes[0]
    
    # m-geodesic (straight in eta)
    p_m = p1 + t * (p2 - p1)
    ax1.plot(t, p_m, 'b-', linewidth=2, label='m-geodesic (alpha=-1)\nStraight in eta space')
    
    # e-geodesic (straight in theta -> transform)
    theta_e = theta1 + t * (theta2 - theta1)
    p_e = theta_to_p(theta_e)
    ax1.plot(t, p_e, 'r-', linewidth=2, label='e-geodesic (alpha=1)\nStraight in theta space')
    
    # Mark midpoints
    ax1.plot(0.5, (p1+p2)/2, 'b^', markersize=12)  # m-midpoint
    ax1.plot(0.5, theta_to_p((theta1+theta2)/2), 'rv', markersize=12)  # e-midpoint
    
    ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    ax1.set_xlabel('t')
    ax1.set_ylabel('p (probability)')
    ax1.set_title('Geodesics in Expectation Parameter Space\nBer(0.2) -> Ber(0.8)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim(0, 1)
    
    # Middle: Natural parameter space
    ax2 = axes[1]
    
    # e-geodesic (straight in theta)
    ax2.plot(t, theta_e, 'r-', linewidth=2, label='e-geodesic (alpha=1)\nStraight in theta space')
    
    # m-geodesic (transform)
    theta_m = p_to_theta(p_m)
    ax2.plot(t, theta_m, 'b-', linewidth=2, label='m-geodesic (alpha=-1)\nStraight in eta space')
    
    ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax2.set_xlabel('t')
    ax2.set_ylabel('theta (natural parameter)')
    ax2.set_title('Geodesics in Natural Parameter Space')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Right: Distribution changes
    ax3 = axes[2]
    
    t_samples = [0, 0.25, 0.5, 0.75, 1.0]
    x_vals = [0, 1]
    
    for i, t_val in enumerate(t_samples):
        idx = int(t_val * 99)
        
        # m-geodesic distribution
        p_m_val = p_m[idx]
        ax3.bar([i-0.2], [1-p_m_val], width=0.35, bottom=0, color='blue', alpha=0.5)
        ax3.bar([i-0.2], [p_m_val], width=0.35, bottom=1-p_m_val, color='blue', alpha=0.8)
        
        # e-geodesic distribution
        p_e_val = p_e[idx]
        ax3.bar([i+0.2], [1-p_e_val], width=0.35, bottom=0, color='red', alpha=0.5)
        ax3.bar([i+0.2], [p_e_val], width=0.35, bottom=1-p_e_val, color='red', alpha=0.8)
    
    ax3.set_xticks(range(5))
    ax3.set_xticklabels(['t=0', 't=0.25', 't=0.5', 't=0.75', 't=1'])
    ax3.set_ylabel('Probability')
    ax3.set_title('Distribution Interpolation\nBlue: m-geodesic, Red: e-geodesic')
    ax3.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
    print("[Alpha-geodesic Characteristics]")
    print(f"Start: Ber({p1}), theta = {theta1:.3f}")
    print(f"End: Ber({p2}), theta = {theta2:.3f}")
    print()
    print("m-geodesic (alpha=-1):")
    print(f"  Midpoint: p = {(p1+p2)/2:.3f} (arithmetic mean)")
    print()
    print("e-geodesic (alpha=1):")
    print(f"  Midpoint: p = {theta_to_p((theta1+theta2)/2):.3f} (arithmetic mean in natural params)")

visualize_alpha_connections()

### 4.4 曲率の可視化

In [ ]:
def visualize_curvature():
    """Curvature of the Gaussian distribution manifold"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Scalar curvature of Gaussian manifold: R = -1 (constant negative curvature)
    # This is isometric to the hyperbolic plane
    
    # Left: Curvature explanation
    ax1 = axes[0]
    
    # Hyperbolic plane (Poincare half-plane model)
    # ds^2 = (dx^2 + dy^2) / y^2
    # Similar to Fisher metric ds^2 = dmu^2/sigma^2 + 2*dsigma^2/sigma^2
    
    # Draw geodesics (semicircles)
    for x_center in [-1, 0, 1, 2]:
        for r in [0.5, 1.0, 1.5]:
            theta = np.linspace(0, np.pi, 100)
            x = x_center + r * np.cos(theta)
            y = r * np.sin(theta)
            ax1.plot(x, y, 'b-', alpha=0.5, linewidth=1)
    
    # Vertical lines (geodesics)
    for x_c in np.linspace(-2, 3, 6):
        ax1.axvline(x_c, color='blue', alpha=0.3, linewidth=1)
    
    ax1.set_xlim(-2, 3)
    ax1.set_ylim(0, 2)
    ax1.set_xlabel('mu')
    ax1.set_ylabel('sigma')
    ax1.set_title('Geodesics on Gaussian Manifold\n(Poincare Half-Plane Model)\nScalar Curvature R = -1')
    ax1.grid(True, alpha=0.3)
    
    # Right: Geodesic triangle and angle defect
    ax2 = axes[1]
    
    # 3 points
    p1 = np.array([0, 1])
    p2 = np.array([1, 0.5])
    p3 = np.array([-0.5, 0.5])
    
    # Geodesics (approximated as semicircles)
    def geodesic_arc(pa, pb, n=50):
        """Compute geodesic (semicircle) between two points"""
        # Simplified: linear approximation (exact would be semicircle)
        t = np.linspace(0, 1, n)
        return pa + t[:, None] * (pb - pa)
    
    arc1 = geodesic_arc(p1, p2)
    arc2 = geodesic_arc(p2, p3)
    arc3 = geodesic_arc(p3, p1)
    
    ax2.plot(arc1[:, 0], arc1[:, 1], 'b-', linewidth=2)
    ax2.plot(arc2[:, 0], arc2[:, 1], 'g-', linewidth=2)
    ax2.plot(arc3[:, 0], arc3[:, 1], 'r-', linewidth=2)
    
    ax2.plot([p1[0], p2[0], p3[0]], [p1[1], p2[1], p3[1]], 'ko', markersize=10)
    
    ax2.set_xlabel('mu')
    ax2.set_ylabel('sigma')
    ax2.set_title('Geodesic Triangle in Negative Curvature Space\nSum of interior angles < pi')
    ax2.set_xlim(-1, 2)
    ax2.set_ylim(0.2, 1.5)
    ax2.grid(True, alpha=0.3)
    
    # Annotation
    ax2.text(0.5, 0.3, 'Negative curvature:\nSum of angles < 180 deg', fontsize=11,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("[Curvature of Gaussian Manifold]")
    print("- Scalar curvature R = -1 (constant negative curvature)")
    print("- Isometric to hyperbolic plane (Poincare half-plane)")
    print("- Geodesics are semicircles or vertical lines")
    print("- Sum of interior angles of geodesic triangle < pi")

visualize_curvature()

## 5. 具体例

### 例1：Christoffel記号の計算

In [ ]:
def calculate_christoffel_gaussian():
    """正規分布のChristoffel記号を計算"""
    print("【正規分布のChristoffel記号】")
    print()
    print("Fisher計量: g = diag(1/σ², 2/σ²)")
    print("逆計量: g⁻¹ = diag(σ², σ²/2)")
    print()
    print("計量の偏微分:")
    print("  ∂g_{μμ}/∂μ = 0")
    print("  ∂g_{μμ}/∂σ = -2/σ³")
    print("  ∂g_{σσ}/∂μ = 0")
    print("  ∂g_{σσ}/∂σ = -4/σ³")
    print()
    print("Christoffel記号の公式:")
    print("  Γᵏᵢⱼ = (1/2) gᵏˡ (∂gᵢˡ/∂xʲ + ∂gⱼˡ/∂xⁱ - ∂gᵢⱼ/∂xˡ)")
    print()
    print("非ゼロ成分:")
    print("  Γᵘμσ = Γᵘσμ = (1/2)σ²·(-2/σ³) = -1/σ")
    print("  Γˢμμ = -(1/2)(σ²/2)·(-2/σ³) = 1/(2σ) ... (訂正: 1/σ)")
    print("  Γˢσσ = (1/2)(σ²/2)·(-4/σ³) = -1/σ")
    print()
    print("測地線方程式:")
    print("  d²μ/dt² - (2/σ)(dμ/dt)(dσ/dt) = 0")
    print("  d²σ/dt² + (1/σ)(dμ/dt)² - (1/σ)(dσ/dt)² = 0")

calculate_christoffel_gaussian()

### 例2：平行移動の計算

In [ ]:
def demonstrate_parallel_transport():
    """Parallel transport along a curve"""
    print("[Parallel Transport Equation]")
    print()
    print("Parallel transport of vector V along curve gamma(t):")
    print("  dV^k/dt + Gamma^k_{ij} (d*gamma^i/dt) V^j = 0")
    print()
    
    # Parallel transport on Gaussian manifold
    def parallel_transport_ode(V, t, gamma_func, dgamma_func, christoffel_func):
        """ODE for parallel transport"""
        gamma = gamma_func(t)
        dgamma = dgamma_func(t)
        Gamma = christoffel_func(gamma)
        
        dV = np.zeros(2)
        for k in range(2):
            for i in range(2):
                for j in range(2):
                    dV[k] -= Gamma[k, i, j] * dgamma[i] * V[j]
        return dV
    
    # Example: parallel transport along horizontal line sigma = 1
    def gamma(t):
        return np.array([t, 1])  # mu = t, sigma = 1
    
    def dgamma(t):
        return np.array([1, 0])  # dmu/dt = 1, dsigma/dt = 0
    
    def christoffel(p):
        sigma = p[1]
        Gamma = np.zeros((2, 2, 2))
        Gamma[0, 0, 1] = -1/sigma
        Gamma[0, 1, 0] = -1/sigma
        Gamma[1, 0, 0] = 1/sigma
        Gamma[1, 1, 1] = -1/sigma
        return Gamma
    
    # Initial vector
    V0 = np.array([0, 1])  # Unit vector in sigma direction
    
    t_span = np.linspace(0, 2, 100)
    V_solution = odeint(parallel_transport_ode, V0, t_span, 
                        args=(gamma, dgamma, christoffel))
    
    print("Example: Parallel transport along horizontal line sigma = 1")
    print(f"Initial vector V(0) = {V0}")
    print(f"After transport V(2) = [{V_solution[-1, 0]:.4f}, {V_solution[-1, 1]:.4f}]")
    print()
    print("In this case, V does not change (Gamma contributions cancel along path)")
    
    # Visualization
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Path
    ax.plot(t_span, np.ones_like(t_span), 'b-', linewidth=2, label='Path gamma(t)')
    
    # Vector evolution
    skip = 10
    for i in range(0, len(t_span), skip):
        ax.arrow(t_span[i], 1, V_solution[i, 0]*0.2, V_solution[i, 1]*0.2,
                 head_width=0.03, head_length=0.02, fc='red', ec='red')
    
    ax.set_xlabel('mu')
    ax.set_ylabel('sigma')
    ax.set_title('Parallel Transport along sigma = 1')
    ax.set_xlim(-0.5, 2.5)
    ax.set_ylim(0.5, 1.5)
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    plt.show()

demonstrate_parallel_transport()

### 例3：双対接続

In [ ]:
def demonstrate_dual_connections():
    """双対接続の関係"""
    print("【双対接続】")
    print()
    print("定義: 接続 ∇ と ∇* が計量 g に関して双対であるとは、")
    print("  X g(Y, Z) = g(∇_X Y, Z) + g(Y, ∇*_X Z)")
    print("が成り立つこと。")
    print()
    print("【α-接続の双対性】")
    print("∇^(α) と ∇^(-α) は Fisher 計量に関して双対")
    print()
    print("特に重要な双対ペア:")
    print("  e-接続 ∇^(1)  ↔  m-接続 ∇^(-1)")
    print()
    print("【情報幾何での意味】")
    print("- e-測地線: 指数型分布族の自然パラメータで直線")
    print("- m-測地線: 混合族の期待値パラメータで直線")
    print("- この双対性が情報幾何の中心的構造")
    print()
    print("【Christoffel記号の関係】")
    print("Γ^(α)ₖᵢⱼ + Γ^(-α)ₖⱼᵢ = ∂gᵢⱼ/∂xᵏ")
    print("（Levi-Civita接続の場合: Γ^(0)ₖᵢⱼ = Γ^(0)ₖⱼᵢ, 自己双対）")

demonstrate_dual_connections()

## 6. 他の概念との関係

### 前の節との繋がり
- **リーマン計量 (§1.5)**: Levi-Civita接続は計量から決まる
- **ベクトル場 (§1.4)**: 共変微分はベクトル場の微分を一般化

### 情報幾何との関連

| 概念 | 一般の微分幾何 | 情報幾何 |
|:---|:---|:---|
| 接続 | $\nabla$ | α-接続 $\nabla^{(\alpha)}$ |
| Christoffel記号 | $\Gamma^k_{ij}$ | $\Gamma^{(\alpha)k}_{ij}$ |
| 測地線 | 最短経路 | α-測地線 |
| 双対接続 | 一般には存在しない | e-接続 ↔ m-接続 |

### 重要な関係式

**α-接続のChristoffel記号**:
$$\Gamma^{(\alpha)k}_{ij} = \Gamma^{(0)k}_{ij} - \frac{\alpha}{2} g^{kl} T_{ijl}$$

**歪度テンソル**:
$$T_{ijk} = E\left[ \partial_i \ell \cdot \partial_j \ell \cdot \partial_k \ell \right]$$

**双対性条件**:
$$g(\nabla^{(\alpha)}_X Y, Z) + g(Y, \nabla^{(-\alpha)}_X Z) = X g(Y, Z)$$

## 7. 演習問題

### Q1. 共変微分の計算

平面上でユークリッド計量を使ったとき、Christoffel記号がすべて0であることを確認せよ。

<details>
<summary>解答を見る</summary>

ユークリッド計量: $g_{ij} = \delta_{ij}$ (定数)

$$\Gamma^k_{ij} = \frac{1}{2} g^{kl} \left( \frac{\partial g_{il}}{\partial x^j} + \frac{\partial g_{jl}}{\partial x^i} - \frac{\partial g_{ij}}{\partial x^l} \right) = 0$$

計量が定数なので、すべての偏微分が0。

</details>

### Q2. ベルヌーイ分布のe-測地線

Ber(0.2) から Ber(0.8) への e-測地線上の t=0.5 での分布を求めよ。

In [ ]:
# Q2検証
def p_to_theta(p):
    return np.log(p / (1 - p))

def theta_to_p(theta):
    return 1 / (1 + np.exp(-theta))

p1, p2 = 0.2, 0.8
theta1, theta2 = p_to_theta(p1), p_to_theta(p2)

# e-測地線の中点（θで線形補間）
theta_mid = 0.5 * theta1 + 0.5 * theta2
p_mid = theta_to_p(theta_mid)

print(f"θ₁ = {theta1:.4f}, θ₂ = {theta2:.4f}")
print(f"θ_mid = {theta_mid:.4f}")
print(f"e-測地線の中点: p = {p_mid:.4f}")
print(f"（参考: 算術平均は p = {(p1+p2)/2:.4f}）")

### Q3. 曲率と平行移動

正規分布多様体のスカラー曲率が $R = -1$ であることから、小さな閉曲線に沿った平行移動でベクトルがどれだけ回転するか説明せよ。

<details>
<summary>解答を見る</summary>

ガウス・ボネの定理より、閉曲線 $C$ に沿った平行移動による回転角 $\Delta\phi$ は：

$$\Delta\phi = \iint_S K \, dA$$

ここで $K$ はガウス曲率（2次元では $K = R/2 = -1/2$）、$S$ は $C$ が囲む領域、$dA$ はFisher計量による面積要素。

負曲率なので、反時計回りの経路では時計回りに回転する。

</details>

## 8. 参考：使用したプロンプト

```
アファイン接続と共変微分の概念を、「曲がった空間での微分」という
観点から直感的に説明してください。ユークリッド空間との違いを強調して。
```

```
正規分布多様体のChristoffel記号を、Fisher計量から計算する過程を
詳細に示してください。
```

```
情報幾何のα-接続について説明してください。特にe-接続とm-接続の
双対性と、それぞれの測地線の特徴を教えてください。
```

```
球面上の平行移動でベクトルが回転することを、Pythonで可視化して
ください。北極から三角形の経路を一周する例を示してください。
```

---
**次のステップ**: 第2章 `ch02_statistical_models/` へ（統計モデルと情報幾何）